# PYNQ 中药气味原始数据识别演示

这是已记录传感器数据的回放，会真实调用 FPGA。没有连接传感器时也能运行。
点击下面代码单元格，再按 **Shift+Enter**。首次运行需要编译 ARM 特征处理库，请等待完成。

演示固定使用第一条单味事件和交付包指定的反序组合示例，没有按识别是否正确挑选。
**硬件结果与参考一致**，和 **预测与真实标签一致** 是两个不同的检查。
训练记录的回放结果不能当作新样本准确率；已知起点由记录提供，没有实现空气或未知药材拒识。


In [ ]:
from pathlib import Path
import subprocess
base = Path.cwd()
assert (base / 'run_raw_on_pynq.sh').is_file(), '请将 Notebook 工作目录切换到部署目录'
run = subprocess.run(['sh', str(base / 'run_raw_on_pynq.sh'), '--examples'],
                     cwd=base, stdin=subprocess.DEVNULL, capture_output=True,
                     text=True, timeout=180)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('识别演示失败，请保留错误信息')


In [ ]:
import json
import pandas as pd
result = json.loads((base / 'raw_results/examples.json').read_text())
table = pd.DataFrame(result['rows'])
display(table[['event_id', 'true_label', 'predicted_label', 'recognition_correct', 'hardware_match']]
        .rename(columns={'event_id':'事件', 'true_label':'真实标签', 'predicted_label':'FPGA 识别结果',
                         'recognition_correct':'识别正确', 'hardware_match':'与整数参考一致'}))


## 已完成的全量回放结果

下面读取已保存的 160 段单味和 410 段组合回放结果，不会再次加载 FPGA。
`existing_training_records` 为现有训练记录回代，`reverse_diagnostic` 为反序诊断，不能合并成现场准确率。
如需重新执行全量回放，在 Terminal 中进入本目录，执行 `sh run_raw_on_pynq.sh --all`。


In [ ]:
if not (base / 'raw_results/all_events.json').exists():
    full = subprocess.run(['sh', str(base / 'run_raw_on_pynq.sh'), '--all'],
                          cwd=base, stdin=subprocess.DEVNULL, capture_output=True,
                          text=True, timeout=600)
    print(full.stdout)
    if full.returncode:
        print(full.stderr)
        raise RuntimeError('全量回放失败，请保留错误信息')
all_results = json.loads((base / 'raw_results/all_events.json').read_text())
display(pd.DataFrame(all_results['summaries']))
print('完整链路与参考一致：', all_results['hardware_matches'], '/', all_results['events'])


结果文件位于本目录的 `raw_results/`：`examples.csv` 和 `all_events.csv` 可下载交给队友。
它们记录实际标签、FPGA 预测、识别是否正确和计算是否一致，不会把错误识别伪装成通过。
下一步需要同型号、同单位、同通道顺序的独立新采集数据及明确的响应起点。
